# Deliverable 4 — Data Cleaning

Two datasets are cleaned here, each following the same pattern: every print
statement documents a decision **and** its consequence (rows/columns affected).
See `README.md` for the full write-up and column-meaning tables.

## 1. Christchurch Airbnb listings

Source: `Airbnb_Oct25_Jun26.csv` — Christchurch listings, Oct 2025 to Jun 2026, built in Deliverable 3.

In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv("Airbnb_Oct25_Jun26.csv")
n_start, c_start = df.shape
print(f"Starting shape: {df.shape}")

Starting shape: (28795, 19)


### Drop columns with zero information content

In [23]:
# 'license' is 100% missing -> carries no information, safe to drop.
# 'neighbourhood_group' has exactly one unique value ("Christchurch City")
#   across all rows -> constant column, zero variance, adds nothing.
zero_info_cols = [c for c in ["license", "neighbourhood_group"] if c in df.columns]
df = df.drop(columns=zero_info_cols)
print(f"Dropped zero-information columns: {zero_info_cols}")

Dropped zero-information columns: ['license', 'neighbourhood_group']


### Drop privacy-sensitive / non-analytical columns

In [24]:
# 'host_name' is free-text, high-cardinality, and not needed for price/
# location analysis (host-level scale is already captured by
# 'calculated_host_listings_count'). 'name' (the listing title) is also
# free text and not needed to compare prices/availability with bond data.
# Keep 'host_id' and 'id' as identifiers in case you need to dedupe or
# trace a listing back later.
drop_text_cols = [c for c in ["host_name", "name"] if c in df.columns]
df = df.drop(columns=drop_text_cols)
print(f"Dropped text/identifying columns: {drop_text_cols}")

Dropped text/identifying columns: ['host_name', 'name']


### Handle missing 'price' (~37% missing)

In [25]:
# A missing price usually means the listing was blocked/unavailable that
# month rather than a data error. Dropping ~37% of rows would badly bias
# next week's price comparison, so instead of deleting rows we flag them.
# Anyone doing price-specific analysis later should filter on this flag.
n_missing_price = df["price"].isna().sum()
df["price_missing"] = df["price"].isna()
print(f"'price' missing in {n_missing_price} rows ({n_missing_price/len(df):.1%}) "
      f"-> kept as NaN + added boolean flag 'price_missing' instead of dropping rows")

'price' missing in 10667 rows (37.0%) -> kept as NaN + added boolean flag 'price_missing' instead of dropping rows


### Handle missing 'reviews_per_month' / 'last_review'

In [26]:
# These are missing together (~9%) and almost certainly mean "0 reviews so
# far", not a data quality problem. number_of_reviews == 0 confirms this.
mask = df["reviews_per_month"].isna() & (df["number_of_reviews"] == 0)
print(f"Rows with missing reviews_per_month AND 0 reviews: {mask.sum()} "
      f"of {df['reviews_per_month'].isna().sum()} missing -> confirms 'no reviews yet', not noise")
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
# last_review has no sensible numeric fill; leave as NaN (means "never reviewed")
# but parse it to a real date so it can be used for recency filtering later.
df["last_review"] = pd.to_datetime(df["last_review"], format="%d/%m/%Y", errors="coerce")

Rows with missing reviews_per_month AND 0 reviews: 2627 of 2627 missing -> confirms 'no reviews yet', not noise


### Parse month_year and enforce dtypes

In [27]:
df["month_year"] = pd.to_datetime(df["month_year"], format="%Y-%m")
df["latitude"] = df["latitude"].astype("float32")
df["longitude"] = df["longitude"].astype("float32")
df["price"] = df["price"].astype("float32")

### Drop exact duplicate rows (same listing, same month)

In [28]:
n_before_dedup = len(df)
df = df.drop_duplicates(subset=["id", "month_year"])
print(f"Removed {n_before_dedup - len(df)} duplicate (id, month_year) rows")

Removed 0 duplicate (id, month_year) rows


### Save — space efficient

In [29]:
df.to_parquet("airbnb_christchurch_clean.parquet", index=False)
# Parquet is columnar + compressed by default (typically 5-10x smaller than
# CSV for this kind of data) and preserves dtypes, so every teammate who
# reruns this script gets an identical file without re-parsing dates.

print(f"\nFinal shape: {df.shape} (started at {(n_start, c_start)})")
print(f"Columns kept: {df.columns.tolist()}")


Final shape: (28795, 16) (started at (28795, 19))
Columns kept: ['id', 'host_id', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365', 'number_of_reviews_ltm', 'month_year', 'price_missing']


## 2. Rental bond data

Source: "Detailed quarterly report, 2020-2026", Tenancy Services (tenancy.govt.nz). `Median Rent` and `Geometric Mean Rent` are related but distinct log-normal-based statistics (see README) - not empirical/positional values. Requires `christchurch_sa2_lookup.csv` in the same folder.

In [30]:
import pandas as pd

df = pd.read_csv("Detailed-Quarterly-Tenancy-Q1-2020-Q3-2026.csv")
n_start, c_start = df.shape
print(f"Starting shape: {df.shape}")

Starting shape: (226080, 12)


### Filter TimeFrame to match the Airbnb dataset's window

In [31]:
# The Airbnb data covers Oct 2025 - Jun 2026 (monthly). This report is
# quarterly, so the matching quarters are the ones whose 3-month window
# overlaps that period: Q4 2025 (Oct-Dec), Q1 2026 (Jan-Mar), Q2 2026 (Apr-Jun).
df["TimeFrame"] = pd.to_datetime(df["TimeFrame"])
overlap_quarters = ["2025-10-01", "2026-01-01", "2026-04-01"]
n_before_time = len(df)
df = df[df["TimeFrame"].isin(pd.to_datetime(overlap_quarters))]
print(f"Filtered TimeFrame to {overlap_quarters} "
      f"-> dropped {n_before_time - len(df)} rows outside this window "
      f"({len(df)} remaining)")

Filtered TimeFrame to ['2025-10-01', '2026-01-01', '2026-04-01'] -> dropped 198868 rows outside this window (27212 remaining)


### Drop rows where Location Id is not a usable, specific area

In [32]:
# -99 is a national rollup (would double-count if summed alongside real
# areas next week); NaN Location Id means the bond couldn't be geocoded at
# all. Neither can be linked to a Christchurch neighbourhood, and together
# they're a small share of rows, so dropping them is low-cost and avoids
# silently corrupting next week's area-level comparison.
n_before_loc = len(df)
n_national = (df["Location Id"] == -99).sum()
n_unmatched = df["Location Id"].isna().sum()
df = df[(df["Location Id"] != -99) & df["Location Id"].notna()]
print(f"Dropped {n_national} national-rollup rows (Location Id == -99) and "
      f"{n_unmatched} unmatched rows (Location Id missing) "
      f"-> {n_before_loc - len(df)} rows removed, {len(df)} remaining")

df["Location Id"] = df["Location Id"].astype("int32")

Dropped 127 national-rollup rows (Location Id == -99) and 94 unmatched rows (Location Id missing) -> 221 rows removed, 26991 remaining


### Filter to Christchurch City only

In [33]:
# Location Id here is a 2018-vintage SA2 code, but the only Stats NZ
# concordance available via Aria links SA2 to Territorial Authority using
# 2023-vintage codes. Most Christchurch SA2s kept the same code across both
# versions (round-hundred codes), so those are used as a "confirmed" list.
# A handful of Location Id values in the Christchurch numeric range (~14% of
# the confirmed volume) don't have a confirmed 2018<->2023 match and are
# EXCLUDED rather than guessed at, to avoid silently mixing in bonds from a
# neighbouring district. See README for the excluded code list if your team
# wants to manually verify and add any of them back in.
chch_lookup = pd.read_csv("christchurch_sa2_lookup.csv")
chch_sa2_codes = set(chch_lookup["sa2_code"])

n_before_chch = len(df)
df = df[df["Location Id"].isin(chch_sa2_codes)]
print(f"Filtered to {len(chch_sa2_codes)} confirmed Christchurch SA2 codes "
      f"-> dropped {n_before_chch - len(df)} rows outside Christchurch "
      f"({len(df)} remaining)")

Filtered to 131 confirmed Christchurch SA2 codes -> dropped 24821 rows outside Christchurch (2170 remaining)


### Missing rent statistics -> leave as NaN (suppressed, not absent)

In [34]:
# These are genuine privacy suppressions from Tenancy Services (small cell
# sizes), not data-quality errors, so imputing a value would misrepresent
# the market. Total/Active/Closed Bonds are still valid for these rows and
# are kept, so volume-based analysis (e.g. "number of available properties")
# isn't affected even where rent stats are blank.
rent_cols = ["Median Rent", "Geometric Mean Rent", "Upper Quartile Rent",
             "Lower Quartile Rent", "Log Std Dev Weekly Rent"]
n_missing_rent = df["Median Rent"].isna().sum()
print(f"{n_missing_rent} of {len(df)} rows have suppressed rent statistics "
      f"-> left as NaN (not imputed); Total/Active/Closed Bonds still usable for these rows")

0 of 2170 rows have suppressed rent statistics -> left as NaN (not imputed); Total/Active/Closed Bonds still usable for these rows


### Dtype cleanup for space efficiency

In [35]:
df["Dwelling Type"] = df["Dwelling Type"].astype("category")
df["Number Of Beds"] = df["Number Of Beds"].astype("category")
for c in ["Total Bonds", "Active Bonds", "Closed Bonds"]:
    df[c] = df[c].astype("int32")
for c in rent_cols:
    df[c] = df[c].astype("float32")

### Save — space efficient

In [36]:
df.to_parquet("bond_data_clean.parquet", index=False)

print(f"\nFinal shape: {df.shape} (started at {(n_start, c_start)})")
print(f"Columns kept: {df.columns.tolist()}")


Final shape: (2170, 12) (started at (226080, 12))
Columns kept: ['TimeFrame', 'Location Id', 'Dwelling Type', 'Number Of Beds', 'Total Bonds', 'Active Bonds', 'Closed Bonds', 'Median Rent', 'Geometric Mean Rent', 'Upper Quartile Rent', 'Lower Quartile Rent', 'Log Std Dev Weekly Rent']
